# Lancer un entraînement

Ce notebook lance **une expérience, celle qu'on lui nomme**, et rend compte de ce qu'elle a
produit. Il n'invente aucun réglage : tout vient du fichier `configs/experiments/<nom>.yaml`, et le
run passe par `run_one`, la fonction que `python -m src.experiments.run` et `make reproduce`
appellent l'un comme l'autre. Ce que le notebook ajoute à la ligne de commande est ce qu'un
terminal ne donne pas : la configuration affichée avant le départ, l'état du poste, les courbes de
perte à l'arrivée, et un noyau où l'enregistrement du run reste sous la main.

Il ne remplace pas `make train-scratch` ni `make train-pretrained`. Une campagne se lance depuis un
terminal, qui survit à la fermeture d'un éditeur. Ce notebook sert le cas inverse : une expérience
à la fois, qu'on regarde tourner.

Deux modes, ceux de `make reproduce` :

- `quick` : deux pas d'optimisation sur un corpus tronqué, quelques secondes. Il vérifie que la
  chaîne tourne sur ce poste. Tout ce qu'il écrit va sous `reports/quick/`, son enregistrement
  porte `PARTIAL` et n'entre dans aucun tableau. **C'est par là qu'on commence.**
- `full` : l'expérience déclarée, telle quelle, aux destinations réelles. De quelques minutes à
  plusieurs heures selon la ligne choisie.

Prérequis : `notebooks/00_environment_check.ipynb` passe, et le corpus est construit
(`make data`). La section 2 ne revérifie que les deux points dont l'absence arrête ce notebook.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sys
from pathlib import Path
from time import perf_counter

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Le repertoire courant, et pas seulement sys.path : une experience lit son
# corpus par le chemin relatif que porte son fichier et ecrit ses checkpoints
# sous runs/. Lance depuis notebooks/, le run creerait notebooks/runs/ et ne
# trouverait pas configs/data/cnn_dailymail.yaml.
os.chdir(REPO_ROOT)

print(f"racine du depot   {REPO_ROOT}")
print(f"interpreteur      {sys.executable}")

## 1. L'expérience à lancer

Les cinq réglages ci-dessous sont les seuls de ce notebook, et aucun ne touche à la science de
l'expérience : ils disent laquelle lancer et où écrire, pas comment entraîner. Un hyperparamètre se
change dans le fichier YAML, où il est versionné, jamais ici.

`EVALUATION_LIMIT` mérite un mot parce que sa conséquence est enregistrée : évaluer les N premiers
documents fait porter au run le statut `PARTIAL`, et un run partiel n'entre dans aucun tableau.
C'est voulu. Un score mesuré sur cinquante documents et un score mesuré sur le jeu de test complet
ne se comparent pas, et rien ne doit permettre au premier de passer pour le second.

`RESUME` reprend l'entraînement depuis les checkpoints déjà présents sous `runs/<nom>/`. Utile
après une interruption, dangereux après un changement de configuration : les poids repris viennent
de l'ancienne, et le run écrira pourtant la nouvelle dans son enregistrement.

Le tableau imprimé donne, pour chaque expérience déclarée, ce que le poste en sait déjà.

In [ ]:
from src.experiments.config import (
    DEFAULT_EXPERIMENTS_DIR,
    ExperimentConfig,
    iter_experiment_files,
    load_experiment_config,
)
from src.experiments.record import STATUS_NOT_RUN, find_record
from src.experiments.run import DEFAULT_RESULTS_DIR, DEFAULT_RUNS_DIR

#: Experience lancee par ce notebook. Un nom declare, pas un chemin.
EXPERIMENT = "scratch_10"

#: quick verifie la chaine et ne produit aucun resultat, full joue l'experience
#: declaree. Meme vocabulaire que make reproduce MODE=...
MODE = "quick"

#: Reprend l'entrainement depuis les checkpoints du repertoire de run.
RESUME = False

#: Envoi du run termine vers MLflow. Un run quick n'est jamais trace.
TRACKING = True

#: Evalue les N premiers documents du split seulement, et marque le run
#: PARTIAL. None evalue le split entier.
EVALUATION_LIMIT: int | None = None

CONFIGS: dict[str, ExperimentConfig] = {}
FILES: dict[str, Path] = {}
for path in iter_experiment_files(DEFAULT_EXPERIMENTS_DIR):
    loaded = load_experiment_config(path)
    CONFIGS[loaded.name] = loaded
    FILES[loaded.name] = path

if EXPERIMENT not in CONFIGS:
    raise SystemExit(
        f"{EXPERIMENT!r} n'est pas declare sous {DEFAULT_EXPERIMENTS_DIR}. "
        f"Noms disponibles : {', '.join(CONFIGS)}"
    )

print(f"{'experience':<24} {'variante':<22} {'corpus':<12} enregistrement")
for name, config in CONFIGS.items():
    found = find_record(DEFAULT_RESULTS_DIR, name)
    percentage = config.dataset.percentage
    proportion = "sans objet" if percentage is None else f"{percentage} %"
    marker = ">" if name == EXPERIMENT else " "
    print(
        f"{marker} {name:<22} {config.variant:<22} {proportion:<12} "
        f"{STATUS_NOT_RUN if found is None else found.status}"
    )

## 2. Le poste et le corpus

Deux vérifications, parce que deux choses seulement peuvent empêcher ce notebook d'aller au bout.
Le contrôle complet de l'environnement est le travail de `00_environment_check.ipynb` ; le répéter
ici ne dirait rien de plus.

Le matériel est lu par `describe_hardware()`, la fonction que chaque run enregistre dans MLflow, de
sorte que la durée observée plus bas reste interprétable à côté de celles de `reports/`. Un poste
sans GPU n'est pas bloqué : il est lent, et la précision mixte n'y sert à rien.

Le corpus, lui, est bloquant. Sa version est ce qui rend deux runs comparables : un score obtenu
sur un autre `dataset_version` ne se compare à rien de ce que le rapport publie. Absent, il arrête
le notebook plutôt que de laisser un entraînement échouer trois cellules plus loin.

In [ ]:
from src.data.config import load_pipeline_config
from src.data.dataset import MANIFEST_NAME, load_manifest
from src.utils.device import describe_hardware, resolve_device

for key, value in describe_hardware().items():
    print(f"{key:<18} {value}")

declared = CONFIGS[EXPERIMENT]
requested = declared.training.device if declared.training else "auto"
device = resolve_device(requested)
print(f"{'peripherique':<18} {device} (demande : {requested})")
print()

pipeline = load_pipeline_config(declared.dataset.config)
processed = pipeline.paths.processed
if not (processed / MANIFEST_NAME).is_file():
    raise SystemExit(f"corpus absent de {processed}. Le construire : make data")

manifest = load_manifest(processed)
print(f"{'corpus':<18} {manifest.name} (source : {manifest.source_dataset})")
print(f"{'version':<18} {manifest.dataset_version[:16]}...")
print(f"{'tokenizer':<18} {manifest.tokenizer}")
for split, size in manifest.splits.items():
    print(f"{'split ' + split:<18} {size} exemples")

## 3. La configuration retenue

Ce qui est imprimé ici est ce que le run va exécuter, champ par champ, et non un résumé rédigé à la
main de ce qu'il devrait exécuter. La différence compte le jour où les deux divergent.

En mode `quick`, `quick_experiment` remplace le budget par celui d'une répétition : une époque,
deux pas, un décodage court. Il ne touche ni à la graine, ni au split, ni à l'architecture, ni à la
proportion déclarée, parce qu'une vérification qui changerait ces valeurs vérifierait une autre
expérience. `max_steps` est ce qui rend le run `PARTIAL`, et les destinations basculent sous
`reports/quick/` et `runs/quick/` : rien de ce qui sort d'une répétition ne peut atterrir là où le
rapport lit ses chiffres.

Un enregistrement déjà présent est signalé. Le run l'écrasera : un score que le code actuel ne sait
plus produire n'est plus un résultat.

In [ ]:
from src.experiments.record import STATUS_OK, run_directory
from src.tracking.store import describe_store
from src.experiments.reproduce import (
    MODE_QUICK,
    MODES,
    QUICK_DESTINATIONS,
    QUICK_EVALUATION_EXAMPLES,
    quick_experiment,
)

if MODE not in MODES:
    raise SystemExit(f"MODE doit valoir l'un de {MODES}, pas {MODE!r}.")

quick = MODE == MODE_QUICK
CONFIG = quick_experiment(declared) if quick else declared
RESULTS_DIR = QUICK_DESTINATIONS.results if quick else DEFAULT_RESULTS_DIR
RUNS_DIR = QUICK_DESTINATIONS.runs if quick else DEFAULT_RUNS_DIR
LIMIT = QUICK_EVALUATION_EXAMPLES if quick else EVALUATION_LIMIT
TRACES = TRACKING and not quick

# Meme calcul que src.experiments.run.execute : c'est lui qui decide du
# repertoire, donc le lire ailleurs qu'ici serait deviner.
PARTIAL = LIMIT is not None or CONFIG.capped
DIRECTORY = run_directory(RESULTS_DIR, CONFIG.name, partial=PARTIAL)

print(f"{CONFIG.name} : {CONFIG.experiment.description}")
print(f"variante          {CONFIG.variant}")
print(f"fichier           {FILES[EXPERIMENT]}")
print(f"graine            {CONFIG.experiment.seed}")
print()

for label, block in (
    ("modele", CONFIG.model),
    ("entrainement", CONFIG.training),
    ("evaluation", CONFIG.evaluation),
):
    if block is None:
        print(f"{label} : aucun bloc, ce run n'entraine pas")
        print()
        continue
    print(f"{label} :")
    for key, value in block.model_dump().items():
        print(f"  {key:<28} {value}")
    print()

if CONFIG.dataset.percentage is not None:
    share = manifest.splits["train"] * CONFIG.dataset.percentage // 100
    print(f"corpus            {CONFIG.dataset.percentage} % declares, environ {share} exemples")
print(f"resultats         {DIRECTORY}")
print(f"checkpoints       {Path(RUNS_DIR) / CONFIG.name}")
print(f"evaluation        {'split entier' if LIMIT is None else f'{LIMIT} premiers documents'}")
print(f"statut attendu    {'PARTIAL' if PARTIAL else STATUS_OK} en cas de succes")
print(f"tracage MLflow    {'oui' if TRACES else 'non'}")
if TRACES:
    # Le magasin est affiche avant le depart, mot de passe masque : un run
    # qui aurait trace dans un fichier local au lieu de la base partagee se
    # voit ici et non apres coup.
    print(f"magasin           {describe_store()}")

previous = find_record(RESULTS_DIR, CONFIG.name)
if previous is not None:
    print()
    print(f"ATTENTION : un enregistrement {previous.status} existe deja, ce run va l'ecraser.")

## 4. L'entraînement

La cellule configure le journal avant de lancer quoi que ce soit, et c'est indispensable : le
`LoggingCallback` publie l'avancement sur le logger `syntra.training`, en `INFO`. Sans racine
configurée, ces enregistrements n'atteignent aucune sortie et un entraînement d'une heure
n'afficherait rien entre sa bannière et son résumé final. `force=True` parce qu'un noyau qui a déjà
exécuté autre chose peut porter des gestionnaires : `basicConfig` ne ferait alors rien du tout.

`run_one` **ne lève pas**. Une expérience qui casse est écrite avec le statut `FAILED` et son
message d'erreur, puis rendue comme les autres. Une cellule qui se termine sans traceback n'est
donc pas une cellule qui a réussi : c'est le statut, en section 5, qui le dit.

Interrompre le noyau arrête la boucle. Les checkpoints déjà écrits sous `runs/<nom>/` restent, et
`RESUME = True` repart de là. En revanche aucun enregistrement n'est produit : un run interrompu
n'est pas un run raté, c'est un run qui n'a pas eu lieu.

Ordre de grandeur, sur le poste de référence du dépôt (RTX 5060 portable) : 125 minutes pour la
campagne complète des neuf expériences, quelques secondes pour une répétition `quick`.

In [ ]:
from src.experiments.reproduce import quick_corpus
from src.experiments.run import format_record, load_corpus, run_one
from src.tracking.client import DEFAULT_EXPERIMENT, build_tracker

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)

tracker = build_tracker(enabled=TRACES, tracking_uri=None, experiment=DEFAULT_EXPERIMENT)

# Le corpus tronque est passe au run plutot que reconstruit : le mode quick
# doit s'entrainer sur une poignee d'exemples sans toucher au corpus fige.
corpus = quick_corpus(load_corpus(CONFIG)) if quick else None

started = perf_counter()
RECORD = run_one(
    CONFIG,
    results_dir=RESULTS_DIR,
    runs_dir=RUNS_DIR,
    limit=LIMIT,
    resume=RESUME,
    tracker=tracker,
    corpus=corpus,
)
elapsed = perf_counter() - started

print()
duration = f"{elapsed:.1f} s" if elapsed < 120 else f"{elapsed / 60:.1f} min"
print(f"{RECORD.experiment} rendu en {duration}, statut {RECORD.status}")

## 5. Ce que le run a produit

`format_record` est la même fonction que celle appelée en fin de ligne de commande : ce bloc est
donc mot pour mot ce qu'aurait affiché `python -m src.experiments.run`. Les scores sont accompagnés
de leur intervalle bootstrap, et un run partiel le dit lui-même plutôt que de laisser lire ses
chiffres comme des résultats.

Les fichiers listés ensuite sont ce qui reste du run sur le disque : l'enregistrement que les
tableaux relisent, les courbes que les figures tracent, les prédictions produites, et les
checkpoints. Ce sont eux qui comptent, pas la sortie de cette cellule.

In [ ]:
from src.experiments.record import STATUS_FAILED

print(format_record(RECORD))

if RECORD.status == STATUS_FAILED:
    # Un echec est enregistre dans le repertoire nominal : il n'y a pas de
    # mesure partielle a isoler, il n'y a pas de mesure du tout.
    DIRECTORY = run_directory(RESULTS_DIR, RECORD.experiment)

print()
print(f"repertoire du run {DIRECTORY}")
for produced in sorted(DIRECTORY.glob("*")) if DIRECTORY.is_dir() else []:
    print(f"  {produced.name:<24} {produced.stat().st_size / 1024:.1f} Kio")

checkpoints = Path(RUNS_DIR) / RECORD.experiment
if checkpoints.is_dir():
    print(f"checkpoints       {checkpoints}")
    for weight in sorted(checkpoints.glob("*.pt")):
        print(f"  {weight.name:<24} {weight.stat().st_size / 1048576:.1f} Mio")

## 6. Les courbes

Elles sont lues dans `history.json`, écrit par le `HistoryCallback` pendant le run. Le notebook ne
recalcule rien : ces listes sont celles que `make figures` trace pour le rapport, à la mise en
forme près.

`matplotlib.pyplot` est importé directement, sans passer par `src.experiments.figures.pyplot()`,
qui fixe le backend `Agg` pour dessiner sans écran. Ici l'écran existe, et forcer `Agg` reviendrait
à ne rien afficher.

La perte par pas montre si l'optimisation avance ; la perte par époque montre où l'écart entre
entraînement et validation se creuse, donc à partir de quand l'arrêt anticipé avait raison. Une
courbe de deux points, en mode `quick`, ne montre ni l'un ni l'autre : elle prouve seulement que la
boucle tourne.

In [ ]:
import matplotlib.pyplot as plt

from src.experiments.record import HISTORY_FILE

history_path = DIRECTORY / HISTORY_FILE
if not history_path.is_file():
    print(f"aucune courbe : {history_path} est absent, ce run n'a pas entraine.")
else:
    curves = json.loads(history_path.read_text(encoding="utf-8"))["curves"]
    step_loss = curves["step_loss"]
    train_loss = curves["train_loss"]
    validation_loss = curves["validation_loss"]

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(range(1, len(step_loss) + 1), step_loss, linewidth=0.8)
    axes[0].set_title("perte par pas d'optimisation")
    axes[0].set_xlabel("pas")
    axes[0].set_ylabel("perte")

    epochs = range(1, len(train_loss) + 1)
    axes[1].plot(epochs, train_loss, marker="o", label="entrainement")
    axes[1].plot(epochs, validation_loss, marker="o", label="validation")
    axes[1].set_title("perte par epoque")
    axes[1].set_xlabel("epoque")
    axes[1].legend()

    for axis in axes:
        axis.grid(alpha=0.3)
    figure.suptitle(f"{RECORD.experiment} ({RECORD.status})")
    figure.tight_layout()
    plt.show()

## 7. Les scores et leurs intervalles

Trois barres, et l'intervalle qui va avec chacune. C'est l'intervalle qui décide de ce qu'on a le
droit de lire : deux scores dont les intervalles se recouvrent ne se classent pas.

Un run `PARTIAL` produit un score sur une poignée de documents. Il est tracé ici pour vérifier que
la chaîne d'évaluation fonctionne, jamais pour être lu comme une mesure.

In [ ]:
from src.metrics.rouge import ROUGE_VARIANTS

report = (RECORD.evaluation or {}).get("report", {})
rouge = report.get("rouge", {})

if not rouge:
    print("aucun score : ce run n'a pas ete evalue.")
else:
    scores = rouge["scores"]
    intervals = rouge.get("intervals", {})

    values = [scores[v]["fmeasure"] for v in ROUGE_VARIANTS]
    lows = [values[i] - intervals.get(v, {}).get("low", values[i]) for i, v in enumerate(ROUGE_VARIANTS)]
    highs = [intervals.get(v, {}).get("high", values[i]) - values[i] for i, v in enumerate(ROUGE_VARIANTS)]

    figure, axis = plt.subplots(figsize=(7, 4))
    bars = axis.bar(ROUGE_VARIANTS, values, color=["#4c72b0", "#dd8452", "#55a868"], width=0.55)
    axis.errorbar(ROUGE_VARIANTS, values, yerr=[lows, highs], fmt="none", ecolor="#333", capsize=6)
    for bar, value in zip(bars, values, strict=True):
        axis.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.4f}",
                  ha="center", va="bottom", fontsize=10)
    axis.set_ylabel("F-mesure")
    axis.set_ylim(0, max(values) * 1.35)
    axis.grid(axis="y", alpha=0.3)
    figure.suptitle(f"{RECORD.experiment} ({RECORD.status}) sur {report.get('size', '?')} documents")
    figure.tight_layout()
    plt.show()

    print(f"{'longueur moyenne':<22} {report['prediction_words']['mean']:.1f} mots produits, "
          f"{report['reference_words']['mean']:.1f} en reference")
    print(f"{'predictions vides':<22} {report['empty_predictions']}")

## 8. Ce que le magasin a reçu

Le run vient d'être écrit sur disque, puis miroité dans MLflow : métadonnées et métriques dans
PostgreSQL, artefacts et poids dans `MLFLOW_ARTIFACT_ROOT`. La cellule suivante relit le magasin
plutôt que de faire confiance au fait que l'appel n'a pas levé.

Le modèle déposé est celui qui a été **évalué**, relu depuis le meilleur checkpoint. Quand l'arrêt
anticipé a retenu une époque antérieure, ce n'est pas l'objet en fin d'entraînement. Seul un run
`OK` entre au registre : un `PARTIAL` a vu deux pas d'optimisation, ses poids ne veulent rien dire.

`make mlflow-ui` sert la même chose dans un navigateur, sur <http://localhost:5000>.

In [ ]:
from src.tracking.model import registered_name

if not TRACES:
    print("run non trace : rien a lire dans le magasin.")
    print("Mettre TRACKING = True et MODE = 'full' pour y deposer un run.")
else:
    import mlflow

    from src.tracking.client import DEFAULT_EXPERIMENT
    from src.tracking.store import resolve_artifact_root, resolve_tracking_uri

    mlflow.set_tracking_uri(resolve_tracking_uri())
    client = mlflow.MlflowClient()
    print(describe_store())
    print()

    experiment = client.get_experiment_by_name(DEFAULT_EXPERIMENT)
    found = client.search_runs(
        [experiment.experiment_id],
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    if not found:
        print(f"aucun run dans l'experience {DEFAULT_EXPERIMENT!r}.")
    else:
        run = found[0]
        print(f"{'run':<22} {run.info.run_name}  ({run.info.run_id})")
        print(f"{'statut':<22} {run.info.status}")
        print(f"{'commit':<22} {run.data.tags.get('git_commit', 'inconnu')[:12]}"
              f"  dirty={run.data.tags.get('git_dirty', '?')}")
        print(f"{'corpus':<22} {run.data.params.get('dataset_version', '?')[:16]}")
        print()
        print("metriques :")
        for key in sorted(run.data.metrics):
            print(f"  {key:<30} {run.data.metrics[key]:.6g}")

    name = registered_name(RECORD.experiment)
    versions = client.search_model_versions(f"name='{name}'")
    print()
    if not versions:
        print(f"registre : aucune version sous {name!r}.")
        print("Attendu pour un run PARTIAL ou FAILED, qui n'est pas une mesure.")
    else:
        latest = max(versions, key=lambda v: int(v.version))
        print(f"registre : {latest.name} version {latest.version}, statut {latest.status}")
        print(f"recharger :  mlflow.pytorch.load_model('models:/{latest.name}/{latest.version}')")

    root = Path(resolve_artifact_root())
    if root.is_dir():
        total = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
        print(f"artefacts : {root} occupe {total / 1048576:.1f} Mio")

## 9. La suite

Un run terminé ne met pas le rapport à jour : cela se demande, et ce notebook ne le déclenche pas
tout seul. La raison est celle qui fait de `make report-sync` une cible distincte de
l'entraînement : régénérer les tableaux du rapport depuis un poste où une seule expérience sur neuf
vient de tourner réécrirait les huit autres en `NOT_RUN`.

Un run `PARTIAL` ou `FAILED` n'ouvre aucune de ces suites. Il n'est pas une mesure, donc il n'entre
dans aucun tableau.

In [ ]:
print("Depuis un terminal, a la racine du depot :")
print()
if RECORD.status == STATUS_OK:
    print("  make report-sync")
    print("  make figures")
else:
    print(f"  le statut {RECORD.status} n'est pas une mesure : ni tableau, ni figure")
    print("  relancer en MODE = 'full' pour produire un resultat")
print("  make mlflow-ui        # http://localhost:5000")

## Ce que ce notebook ne fait pas

Il ne construit pas le corpus. `make data` télécharge CNN/DailyMail et fige le sous-ensemble de
travail ;
c'est long, c'est en réseau, et cela ne doit pas se déclencher par surprise au milieu d'une
cellule.

Il ne lance pas la campagne. Les neuf expériences, les ablations, les tableaux et les figures sont
l'affaire de `make reproduce MODE=full`, dans un terminal qui survit à la fermeture de l'éditeur.
Ici, une expérience à la fois.

Il ne compare rien. Un score s'y lit seul, sans les autres lignes du tableau ni l'intervalle des
runs voisins. La comparaison entre les deux familles de modèles est le travail du rapport, qui la
tire des enregistrements de tous les runs et non du dernier.